
<div class="alert alert-block alert-info">  
    
## Import & Connections
</div>

In [57]:
import numpy as np
import pandas as pd
import hashlib
import glob
import time
import os
import re
import matplotlib.pyplot as plt

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [58]:
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_columns', 50)

In [59]:
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [68]:
start_date = '20250122'
end_date = '20250122'
response_type = 'ONLINE_SALES'
pagename = 'PostOrderScreen'
campaign_name = 'Exp1_MultiCard_Rapido'
format = 'multiCardBanner'

In [65]:
print(os.getcwd())
local_dataset = '/Users/E2074/local-datasets/customer/ads-monetisation/page-format-experiment/multi-card-banner'
print(local_dataset)

/Users/E2074/analytics/customer/Ads-Monetisation/page-format-experiment
/Users/E2074/local-datasets/customer/ads-monetisation/page-format-experiment/multi-card-banner


<div class="alert alert-block alert-warning">  
    
## Datasets & Functions
</div>

In [69]:
start = datetime.strptime(start_date, '%Y%m%d')
end = datetime.strptime(end_date, '%Y%m%d')

# Generate the list of dates
date_range = [(start + timedelta(days=i)).strftime('%Y%m%d') for i in range((end - start).days + 1)]
print(date_range)

['20250122']


In [71]:
def get_customer_metadata(date_range):

    all_df = []
    
    for date in date_range:
    
        customer_metadata = f"""
            
            with response as (
    
                select
                    distinct
                    yyyymmdd,
                    json_extract_scalar(metadata1, '$.campaignName') AS campaignName,
                    json_extract_scalar(metadata1, '$.slotUnitId') AS slotUnitId,
                    city,
                    os,
                    userid,
                    service,
                    orderstatus,
                    totalcardcount,
                    cardposition,
                    coalesce(aduuid, adid) response_id
                from 
                    canonical.iceberg_log_telemetry_ads_impressions_immutable
                where 
                    yyyymmdd = '{date}'
                    and os = 'android'
                    and responsetype = '{response_type}'
                    and eventname = 'Ad_Response'
                    and pagename = '{pagename}'
                    and format = '{format}'
                    and regexp_like(metadata1, '{campaign_name}')
                ),
            
                impression as (
                
                select
                    distinct
                    yyyymmdd,
                    userid,
                    cardposition,
                    coalesce(aduuid, adid) impression_id
                from 
                    canonical.iceberg_log_telemetry_ads_impressions_immutable
                where 
                    yyyymmdd = '{date}'
                    and os = 'android'
                    and responsetype = '{response_type}'
                    and eventname = 'Ad_Impression'
                    and pagename = '{pagename}'
                    and format = '{format}'
                    and regexp_like(metadata1, '{campaign_name}')
                ),
                
                viewable_impression as (
                
                select
                    distinct
                    yyyymmdd,
                    userid,
                    cardposition,
                    coalesce(aduuid, adid) viewable_impression_id
                from 
                    canonical.iceberg_log_telemetry_ads_impressions_immutable
                where 
                    yyyymmdd = '{date}'
                    and os = 'android'
                    and responsetype = '{response_type}'
                    and eventname = 'Ad_Viewable_Impression'
                    and pagename = '{pagename}'
                    and format = '{format}'
                    and regexp_like(metadata1, '{campaign_name}')
                ),
                
                click as (
                
                select
                    distinct
                    yyyymmdd,
                    userid,
                    cardposition,
                    coalesce(aduuid, adid) click_id
                from 
                    canonical.iceberg_log_telemetry_ads_impressions_immutable
                where 
                    yyyymmdd = '{date}'
                    and os = 'android'
                    and responsetype = '{response_type}'
                    and eventname = 'Ad_Click'
                    and pagename = '{pagename}'
                    and format = '{format}'
                )
                
                select
                    a.*,
                    impression_id,
                    coalesce(click_id, viewable_impression_id) viewable_impression_id,
                    click_id
                from 
                    response a
                left join 
                    impression b
                    on a.yyyymmdd = b.yyyymmdd
                    and a.userid = b.userid
                    and a.response_id = b.impression_id
                    and a.cardposition = b.cardposition
                left join 
                    viewable_impression c
                    on b.yyyymmdd = c.yyyymmdd
                    and b.userid = c.userid
                    and b.impression_id = c.viewable_impression_id
                    and b.cardposition = c.cardposition
                left join 
                    click d
                    on b.yyyymmdd = d.yyyymmdd
                    and b.userid = d.userid
                    and b.impression_id = d.click_id
                    and b.cardposition = d.cardposition
        """
        
        df = pd.read_sql(customer_metadata, connection)
        all_df.append(df)
        df.to_parquet(local_dataset + '/data_dump/Exp1_MultiCard_Rapido_{}.parquet'.format(date), index=False)
    
    # Concatenate all DataFrames
    final_df = pd.concat(all_df, ignore_index=True)

    return final_df


df_data = get_customer_metadata(date_range)

DatabaseError: {'message': 'Query exceeded the maximum execution time limit of 10.00m', 'errorCode': 131075, 'errorName': 'EXCEEDED_TIME_LIMIT', 'errorType': 'INSUFFICIENT_RESOURCES', 'failureInfo': {'type': 'io.trino.spi.TrinoException', 'message': 'Query exceeded the maximum execution time limit of 10.00m', 'suppressed': [], 'stack': ['io.trino.execution.QueryTracker.enforceTimeLimits(QueryTracker.java:187)', 'io.trino.execution.QueryTracker.lambda$start$0(QueryTracker.java:89)', 'java.base/java.util.concurrent.Executors$RunnableAdapter.call(Executors.java:515)', 'java.base/java.util.concurrent.FutureTask.runAndReset(FutureTask.java:305)', 'java.base/java.util.concurrent.ScheduledThreadPoolExecutor$ScheduledFutureTask.run(ScheduledThreadPoolExecutor.java:305)', 'java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.java:1128)', 'java.base/java.util.concurrent.ThreadPoolExecutor$Worker.run(ThreadPoolExecutor.java:628)', 'java.base/java.lang.Thread.run(Thread.j

In [9]:
def get_customer_segment():    
    
    customer_segment = f"""
    
        with response as (

            select
                distinct
                userid
            from 
                canonical.iceberg_log_telemetry_ads_impressions_immutable
            where 
                yyyymmdd >= '{start_date}'
                and yyyymmdd <= '{end_date}'
                and os = 'android'
                and responsetype = '{response_type}'
                and eventname = 'Ad_Response'
                and pagename = '{pagename}'
                and format = '{format}'
                and regexp_like(metadata1, '{campaign_name}')
            ),

            segment as (
    
            select
                customer_id,
                gender_tag,
                taxi_ltr_segment,
                taxi_retention_segments,
                taxi_regularity_segment,
                taxi_fe_regularity_segment,
                taxi_income_segment,
                taxi_service_affinity,
                taxi_gc_tag
            from 
                datasets.iallocator_customer_segments
            where 
                run_date = '2025-01-14'
            )
            
            select
                userid,
                coalesce(gender_tag, 'UNKNOWN') gender,
                coalesce(taxi_ltr_segment, 'NEW') ltr_segment,
                coalesce(taxi_retention_segments, 'UNKNOWN') retention_segments,
                coalesce(taxi_regularity_segment, 'UNKNOWN') regularity_segment,
                coalesce(taxi_fe_regularity_segment, 'UNKNOWN') fe_regularity_segment,
                coalesce(taxi_income_segment, 'UNKNOWN') income_segment,
                coalesce(taxi_service_affinity, 'UNKNOWN') service_affinity,
                coalesce(taxi_gc_tag, 'UNKNOWN') gc_tag
            from 
                response
            left join 
                segment
                on userid = customer_id
            
    """
    
    df = pd.read_sql(customer_segment, connection)
    df.to_parquet(local_dataset + '/customer_segment.parquet', index=False)

    return df

df_data = get_customer_segment()


<div class="alert alert-block alert-info">  
    
## Load data
</div>

In [7]:
def get_customer_metadata():
    
    # Use glob to find all Parquet files in the directory
    parquet_files = glob.glob(os.path.join(local_dataset + '/data_dump/', "*.parquet"))
    customer_metadata = pd.concat([pd.read_parquet(file) for file in parquet_files], ignore_index=True)

    customer_segment = pd.read_parquet(local_dataset + '/customer_segment.parquet')
    
    return customer_metadata, customer_segment

df_cux_metadata, df_cux_segment = get_customer_metadata()

In [8]:
df_cux_metadata.shape, df_cux_segment.shape

((11605968, 14), (1328444, 9))

In [9]:
df_cux_metadata.head(1)

,yyyymmdd,campaignName,slotUnitId,city,os,userid,service,orderstatus,totalcardcount,cardposition,response_id,impression_id,viewable_impression_id,click_id
0,20250119,Exp1_MultiCard_Rapido2,PostOrderStarted4,Hyderabad,android,5e66e3a094fa3529db220e0c,Auto,started,4,2,62c8eaa5-e329-494d-a289-90e18d7d34155e66e3a094...,None,None,None


In [10]:
df_cux_segment.head(1)

,userid,gender,ltr_segment,retention_segments,regularity_segment,fe_regularity_segment,income_segment,service_affinity,gc_tag
0,6739d9c097e2dfb503918a15,FEMALE,UNKNOWN,UNKNOWN,UNKNOWN,RARE_NEED,MEDIUM_INCOME,UNKNOWN,UNKNOWN


In [11]:
df = pd.merge(df_cux_metadata, df_cux_segment, how='left', on = 'userid')
df.head(1).T

,0
yyyymmdd,20250119
campaignName,Exp1_MultiCard_Rapido2
slotUnitId,PostOrderStarted4
city,Hyderabad
os,android
userid,5e66e3a094fa3529db220e0c
service,Auto
orderstatus,started
totalcardcount,4
cardposition,2


<div class="alert alert-block alert-warning">  
    
## Funnel Func
</div>

In [40]:

def get_funnel(df, groupby):
    
    df_temp = df\
                .groupby(groupby + ['cardposition'])\
                .agg(responses = ('response_id', 'nunique'),
                     impression = ('impression_id', 'nunique'),
                     viewable_impression = ('viewable_impression_id', 'nunique'),
                     click = ('click_id', 'nunique'),
                    )\
                .reset_index()

    df_temp['% render rate'] = (df_temp['impression']*100.00/df_temp['responses']).round(2)
    df_temp['% view'] = (df_temp['viewable_impression']*100.00/df_temp['impression']).round(2)
    df_temp['% ctr'] = (df_temp['click']*100.00/df_temp['viewable_impression']).round(2)

    df_temp['format'] = format
    
    columns = ['format', 'cardposition'] + groupby + ['responses', '% render rate', '% view', '% ctr']
    
    return df_temp[columns].sort_values(by='% ctr', ascending=False)

<div class="alert alert-block alert-success">  
    
## Analysis
</div>

In [13]:
df.head(5)

,yyyymmdd,campaignName,slotUnitId,city,os,userid,service,orderstatus,totalcardcount,cardposition,response_id,impression_id,viewable_impression_id,click_id,gender,ltr_segment,retention_segments,regularity_segment,fe_regularity_segment,income_segment,service_affinity,gc_tag
0,20250119,Exp1_MultiCard_Rapido2,PostOrderStarted4,Hyderabad,android,5e66e3a094fa3529db220e0c,Auto,started,4,2,62c8eaa5-e329-494d-a289-90e18d7d34155e66e3a094...,None,None,None,MALE,PHH,ELITE,WEEKLY,WEEKLY,UNKNOWN,ONLY_LINK,NON_GC
1,20250119,Exp1_MultiCard_Rapido4,PostOrderStarted4,Hyderabad,android,61ac4e8c17dd6a53af0a82d7,Auto,started,4,2,6aa05a84-5147-4ead-99f8-de9d0def041e61ac4e8c17...,None,None,None,FEMALE,PHH,PLATINUM,WEEKLY,BI_WEEKLY,HIGH_INCOME,ONLY_AUTO,NON_GC
2,20250119,Exp1_MultiCard_Rapido2,PostOrderStarted4,Chennai,android,5e03b94a8669fa330a3c844b,Auto,started,4,2,1fb0cde9-f48b-4b0a-be8c-f595308f0c9e5e03b94a86...,None,None,None,MALE,HH,HH,UNKNOWN,BI_WEEKLY,LOW_INCOME,UNKNOWN,UNKNOWN
3,20250119,Exp1_MultiCard_Rapido1,PostOrderStarted4,Delhi,android,6312ef7850e9ee18f1885bd7,Link,started,4,1,0091b457-4faa-4835-a628-a53f36d30b7a6312ef7850...,None,None,None,MALE,PHH,PLATINUM,WEEKLY,BI_WEEKLY,HIGH_INCOME,ONLY_LINK,NON_GC
4,20250119,Exp1_MultiCard_Rapido3,PostOrderOnTheWay4,Pune,android,678c197f4fec9dd974409cd1,CabEconomy,onTheWay,4,0,94cdbb2a-6b5e-4b94-9d61-8d8ad34243f1678c197f4f...,94cdbb2a-6b5e-4b94-9d61-8d8ad34243f1678c197f4f...,94cdbb2a-6b5e-4b94-9d61-8d8ad34243f1678c197f4f...,None,UNKNOWN,NEW,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN


#### Bivariate data 

In [41]:
display(get_funnel(df, ['os']))

,format,cardposition,os,responses,% render rate,% view,% ctr
0,multiCardBanner,0,android,3264495,82.53,73.63,0.25
1,multiCardBanner,1,android,3007092,0.00,NaN,NaN
2,multiCardBanner,2,android,2967578,0.00,NaN,NaN
3,multiCardBanner,3,android,2366799,0.00,NaN,NaN


In [42]:
display(get_funnel(df, ['yyyymmdd']))

,format,cardposition,yyyymmdd,responses,% render rate,% view,% ctr
12,multiCardBanner,0,20250119,881433,83.05,73.16,0.28
16,multiCardBanner,0,20250120,861477,82.23,73.54,0.27
8,multiCardBanner,0,20250118,864586,82.69,74.59,0.25
20,multiCardBanner,0,20250121,656789,82.00,73.11,0.19
0,multiCardBanner,0,20250116,3,100.00,100.00,0.00
4,multiCardBanner,0,20250117,207,88.89,70.11,0.00
1,multiCardBanner,1,20250116,3,0.00,NaN,NaN
2,multiCardBanner,2,20250116,3,0.00,NaN,NaN
3,multiCardBanner,3,20250116,4,0.00,NaN,NaN
5,multiCardBanner,1,20250117,189,0.00,NaN,NaN


In [43]:
display(get_funnel(df, ['slotUnitId']))

,format,cardposition,slotUnitId,responses,% render rate,% view,% ctr
8,multiCardBanner,0,PostOrderStarted4,791399,80.69,77.32,0.48
4,multiCardBanner,0,PostOrderOnTheWay4,1729282,83.99,68.79,0.19
0,multiCardBanner,0,PostOrderArrived4,743814,81.09,81.37,0.14
1,multiCardBanner,1,PostOrderArrived4,673846,0.00,NaN,NaN
2,multiCardBanner,2,PostOrderArrived4,662115,0.00,NaN,NaN
3,multiCardBanner,3,PostOrderArrived4,527369,0.00,NaN,NaN
5,multiCardBanner,1,PostOrderOnTheWay4,1614980,0.00,NaN,NaN
6,multiCardBanner,2,PostOrderOnTheWay4,1600607,0.00,NaN,NaN
7,multiCardBanner,3,PostOrderOnTheWay4,1273652,0.00,NaN,NaN
9,multiCardBanner,1,PostOrderStarted4,718266,0.00,NaN,NaN


In [44]:
display(get_funnel(df, ['service']))

,format,cardposition,service,responses,% render rate,% view,% ctr
4,multiCardBanner,0,Auto C2C,690,82.03,69.08,0.77
16,multiCardBanner,0,Auto Pool,547,83.55,72.65,0.60
32,multiCardBanner,0,Bike Pink,255,84.31,81.86,0.57
36,multiCardBanner,0,C2C,14079,81.04,74.34,0.44
12,multiCardBanner,0,Auto Pet,544,82.54,56.35,0.40
24,multiCardBanner,0,Bike Lite,241258,84.17,76.93,0.39
48,multiCardBanner,0,E rickshaw,3908,82.68,75.21,0.33
44,multiCardBanner,0,CabPremium,50645,81.21,65.35,0.29
52,multiCardBanner,0,Link,1165194,83.38,74.90,0.26
0,multiCardBanner,0,Auto,1192141,82.24,73.27,0.25


In [45]:
display(get_funnel(df, ['gender']))

,format,cardposition,gender,responses,% render rate,% view,% ctr
0,multiCardBanner,0,FEMALE,918151,83.08,74.57,0.29
12,multiCardBanner,0,UNKNOWN,256367,84.51,72.19,0.29
4,multiCardBanner,0,MALE,2078220,82.02,73.41,0.23
8,multiCardBanner,0,OTHERS,11757,85.10,70.80,0.21
1,multiCardBanner,1,FEMALE,845721,0.00,NaN,NaN
2,multiCardBanner,2,FEMALE,834827,0.00,NaN,NaN
3,multiCardBanner,3,FEMALE,661841,0.00,NaN,NaN
5,multiCardBanner,1,MALE,1910863,0.00,NaN,NaN
6,multiCardBanner,2,MALE,1884270,0.00,NaN,NaN
7,multiCardBanner,3,MALE,1506949,0.00,NaN,NaN


In [46]:
display(get_funnel(df, ['ltr_segment']))

,format,cardposition,ltr_segment,responses,% render rate,% view,% ctr
4,multiCardBanner,0,NEW,189975,85.21,71.50,0.33
12,multiCardBanner,0,UNKNOWN,45355,84.89,66.06,0.33
0,multiCardBanner,0,HH,329620,83.47,72.31,0.27
8,multiCardBanner,0,PHH,2699545,82.18,74.08,0.24
1,multiCardBanner,1,HH,306718,0.00,NaN,NaN
2,multiCardBanner,2,HH,303012,0.00,NaN,NaN
3,multiCardBanner,3,HH,244645,0.00,NaN,NaN
5,multiCardBanner,1,NEW,178201,0.00,NaN,NaN
6,multiCardBanner,2,NEW,176939,0.00,NaN,NaN
7,multiCardBanner,3,NEW,140679,0.00,NaN,NaN


In [47]:
display(get_funnel(df, ['retention_segments']))

,format,cardposition,retention_segments,responses,% render rate,% view,% ctr
32,multiCardBanner,0,UNKNOWN,235330,85.15,70.45,0.33
8,multiCardBanner,0,GOLD,442838,82.21,73.82,0.26
12,multiCardBanner,0,HH,146317,83.33,72.63,0.26
4,multiCardBanner,0,ELITE,923083,82.34,74.49,0.25
16,multiCardBanner,0,INACTIVE,204246,82.93,72.56,0.25
0,multiCardBanner,0,DORMANT,451062,82.21,73.29,0.24
20,multiCardBanner,0,PLATINUM,611512,82.14,74.29,0.23
28,multiCardBanner,0,SILVER,218077,82.10,73.35,0.23
24,multiCardBanner,0,PRIME,32030,81.59,75.99,0.22
1,multiCardBanner,1,DORMANT,416004,0.00,NaN,NaN


In [48]:
display(get_funnel(df, ['regularity_segment']))

,format,cardposition,regularity_segment,responses,% render rate,% view,% ctr
16,multiCardBanner,0,UNKNOWN,564950,84.17,71.53,0.30
12,multiCardBanner,0,RARE_NEED,307837,82.48,73.75,0.26
0,multiCardBanner,0,BI_WEEKLY,564247,82.12,73.87,0.25
4,multiCardBanner,0,DAILY,779759,82.39,74.43,0.25
20,multiCardBanner,0,WEEKLY,752910,81.88,74.13,0.23
8,multiCardBanner,0,MONTHLY,294792,82.23,73.77,0.22
1,multiCardBanner,1,BI_WEEKLY,519198,0.00,NaN,NaN
2,multiCardBanner,2,BI_WEEKLY,512403,0.00,NaN,NaN
3,multiCardBanner,3,BI_WEEKLY,411710,0.00,NaN,NaN
5,multiCardBanner,1,DAILY,714494,0.00,NaN,NaN


In [49]:
display(get_funnel(df, ['fe_regularity_segment']))

,format,cardposition,fe_regularity_segment,responses,% render rate,% view,% ctr
16,multiCardBanner,0,UNKNOWN,617373,83.64,72.64,0.29
4,multiCardBanner,0,DAILY,835115,82.34,74.30,0.25
0,multiCardBanner,0,BI_WEEKLY,733822,82.15,73.77,0.24
8,multiCardBanner,0,MONTHLY,423862,82.47,73.39,0.24
12,multiCardBanner,0,RARE_NEED,159377,82.94,72.67,0.24
20,multiCardBanner,0,WEEKLY,494946,81.94,74.06,0.22
1,multiCardBanner,1,BI_WEEKLY,675366,0.00,NaN,NaN
2,multiCardBanner,2,BI_WEEKLY,666025,0.00,NaN,NaN
3,multiCardBanner,3,BI_WEEKLY,534468,0.00,NaN,NaN
5,multiCardBanner,1,DAILY,765177,0.00,NaN,NaN


In [50]:
display(get_funnel(df, ['income_segment']))

,format,cardposition,income_segment,responses,% render rate,% view,% ctr
4,multiCardBanner,0,LOW_INCOME,177038,89.56,77.16,0.30
12,multiCardBanner,0,UNKNOWN,780976,81.53,72.26,0.29
8,multiCardBanner,0,MEDIUM_INCOME,1062045,88.18,76.26,0.24
0,multiCardBanner,0,HIGH_INCOME,1244436,77.33,71.40,0.22
1,multiCardBanner,1,HIGH_INCOME,1106558,0.00,NaN,NaN
2,multiCardBanner,2,HIGH_INCOME,1081279,0.00,NaN,NaN
3,multiCardBanner,3,HIGH_INCOME,857795,0.00,NaN,NaN
5,multiCardBanner,1,LOW_INCOME,170831,0.00,NaN,NaN
6,multiCardBanner,2,LOW_INCOME,170428,0.00,NaN,NaN
7,multiCardBanner,3,LOW_INCOME,136603,0.00,NaN,NaN


In [51]:
display(get_funnel(df, ['service_affinity']))

,format,cardposition,service_affinity,responses,% render rate,% view,% ctr
8,multiCardBanner,0,LINK_CAB,17045,81.80,71.30,0.29
24,multiCardBanner,0,ONLY_LINK,744047,83.45,75.58,0.26
28,multiCardBanner,0,UNKNOWN,1517697,82.75,72.98,0.26
16,multiCardBanner,0,ONLY_AUTO,558468,81.83,73.61,0.25
4,multiCardBanner,0,LINK_AUTO,97201,83.18,74.77,0.24
12,multiCardBanner,0,NO_AFFINITY,186192,81.31,72.91,0.22
20,multiCardBanner,0,ONLY_CAB,106794,79.37,70.14,0.17
0,multiCardBanner,0,AUTO_CAB,37051,79.15,72.30,0.16
1,multiCardBanner,1,AUTO_CAB,33321,0.00,NaN,NaN
2,multiCardBanner,2,AUTO_CAB,32766,0.00,NaN,NaN


In [52]:
display(get_funnel(df, ['gc_tag']))

,format,cardposition,gc_tag,responses,% render rate,% view,% ctr
0,multiCardBanner,0,GC,65796,82.41,75.65,0.26
8,multiCardBanner,0,UNKNOWN,2620084,82.78,73.55,0.26
4,multiCardBanner,0,NON_GC,578615,81.40,73.76,0.22
1,multiCardBanner,1,GC,60341,0.00,NaN,NaN
2,multiCardBanner,2,GC,59497,0.00,NaN,NaN
3,multiCardBanner,3,GC,46886,0.00,NaN,NaN
5,multiCardBanner,1,NON_GC,529440,0.00,NaN,NaN
6,multiCardBanner,2,NON_GC,521599,0.00,NaN,NaN
7,multiCardBanner,3,NON_GC,417324,0.00,NaN,NaN
9,multiCardBanner,1,UNKNOWN,2417311,0.00,NaN,NaN
